# ComfortPy Simple Tutorial — Core Features

**No optional extras needed — just numpy, pandas, and pythermalcomfort.**

This notebook covers the complete core ComfortPy workflow:

1. **Load** sensor data with `SensorData`
2. **Evaluate** each domain (thermal, visual, acoustic, IAQ)
3. **Combine** into a Global IEQ Index (0–100)
4. **Weighting** presets for different building types
5. **Compliance** and contract-ready reports
6. **Missing domains** — graceful weight renormalization

## 1. Load Sensor Data

`SensorData` wraps a pandas DataFrame and auto-detects column names using built-in aliases (`DEFAULT_ALIASES`). It validates physical bounds and handles NaNs.

In [1]:
import numpy as np
import pandas as pd
from comfortpy import SensorData

# Simulate 8 hours of sensor readings (1 per hour)
n = 8
rng = np.random.default_rng(seed=42)

df = pd.DataFrame({
    "temperature": 22 + rng.normal(0, 1, n),       # air temp °C
    "radiant_temp": 22 + rng.normal(0, 1, n),       # mean radiant temp °C
    "rh": 50 + rng.normal(0, 5, n),                 # relative humidity %
    "air_speed": 0.1 + rng.normal(0, 0.02, n),      # air velocity m/s
    "lux": 450 + rng.normal(0, 50, n),              # illuminance lux
    "laeq": 45 + rng.normal(0, 5, n),               # noise L_Aeq dB
    "co2": 700 + rng.normal(0, 100, n),             # CO₂ ppm
})

df

,temperature,radiant_temp,rh,air_speed,lux,laeq,co2
0,22.304717,21.983199,51.843754,0.091433,424.387864,48.716271,767.891356
1,20.960016,21.146956,45.205587,0.092957,409.311364,47.715771,706.757907
2,22.750451,22.879398,54.392252,0.110646,480.798971,41.672451,728.911940
3,22.940565,22.777792,49.750370,0.107309,506.448615,46.160807,763.128823
4,20.048965,22.066031,49.075688,0.108255,444.302627,45.583429,554.284418
5,20.697820,23.127241,46.595352,0.108616,407.992176,46.093443,668.032878
6,22.127840,22.467509,56.112707,0.142833,408.775939,49.357144,652.962735
7,21.683757,21.140708,49.227353,0.091872,482.529639,46.117978,636.112215


In [2]:
# Wrap with SensorData — auto-detects columns via DEFAULT_ALIASES
sensor = SensorData(df)

print("Detected column mapping:")
for original, canonical in sensor.column_map.items():
    print(f"  '{original}' -> '{canonical}'")

print(f"\nAvailable IEQ domains: {sensor.available_domains()}")

# Validate (checks physical bounds, handles NaNs)
sensor.validate(nan_strategy="interpolate")
print("Validation: OK")

Detected column mapping:
  'air_temp_c' -> 'temperature'
  'relative_humidity_pct' -> 'rh'
  'air_velocity_ms' -> 'air_speed'
  'illuminance_lux' -> 'lux'
  'co2_ppm' -> 'co2'
  'noise_laeq_db' -> 'laeq'

Available IEQ domains: ['visual', 'acoustic', 'iaq']
Validation: OK


## 2. Evaluate Individual Domains

Each domain function takes NumPy arrays (or scalars) and returns a result dataclass with scores, compliance flags, and metadata.

In [3]:
from comfortpy import evaluate_thermal, evaluate_visual, evaluate_acoustic, evaluate_iaq

# --- Thermal (ISO 7730 / ASHRAE 55) ---
# Requires: air temp, radiant temp, air velocity, humidity, metabolic rate, clothing
thermal = evaluate_thermal(
    tdb=df["temperature"].values,
    tr=df["radiant_temp"].values,
    vr=df["air_speed"].values,
    rh=df["rh"].values,
    met=1.2,    # sedentary office work
    clo=0.5,    # typical summer clothing
    category="B",  # ISO 7730 category B (normal expectations)
)

print("Thermal (ISO 7730 Category B):")
print(f"  PMV:       {np.round(thermal.pmv, 2)}")
print(f"  PPD:       {np.round(thermal.ppd, 1)}%")
print(f"  Compliant: {thermal.compliant}  (PMV in [-0.5, +0.5])")
print(f"  Category:  {thermal.category}")

Thermal (ISO 7730 Category B):
  PMV:       [-0.75 -1.14 -0.58 -0.58 -1.16 -0.92 -0.85 -0.99]
  PPD:       [16.8 32.2 12.1 12.  33.4 22.8 20.3 25.5]%
  Compliant: [False False False False False False False False]  (PMV in [-0.5, +0.5])
  Category:  B


In [4]:
# --- Visual (EN 12464-1) ---
# Requires: illuminance (lux), task type
visual = evaluate_visual(
    illuminance=df["lux"].values,
    task_type="office_writing",  # target: 500 lux
)

print(f"Visual (EN 12464-1, office_writing = {visual.target_lux} lux target):")
print(f"  Illuminance: {np.round(visual.illuminance, 0)} lux")
print(f"  Score:       {np.round(visual.score, 1)}")
print(f"  Compliant:   {visual.compliant}  (illuminance >= target)")

Visual (EN 12464-1, office_writing = 500.0 lux target):
  Illuminance: [424. 409. 481. 506. 444. 408. 409. 483.] lux
  Score:       [84.9 81.9 96.2 99.7 88.9 81.6 81.8 96.5]
  Compliant:   [False False False  True False False False False]  (illuminance >= target)


In [5]:
# --- Acoustic (Noise Criteria) ---
# Requires: L_Aeq (dB), NC level
acoustic = evaluate_acoustic(
    laeq=df["laeq"].values,
    nc_level="NC-35",  # suitable for open-plan offices
)

print(f"Acoustic (NC-35, threshold = {acoustic.threshold_db} dBA):")
print(f"  L_Aeq:      {np.round(acoustic.laeq, 1)} dB")
print(f"  Score:       {np.round(acoustic.score, 1)}")
print(f"  Compliant:   {acoustic.compliant}  (L_Aeq <= threshold)")

# --- IAQ (CO₂-based ventilation adequacy) ---
# Requires: CO₂ (ppm), threshold level
iaq = evaluate_iaq(
    co2=df["co2"].values,
    threshold_level="good",  # 1000 ppm threshold
)

print(f"\nIAQ (threshold 'good' = {iaq.threshold_ppm} ppm):")
print(f"  CO₂:        {np.round(iaq.co2, 0)} ppm")
print(f"  Score:       {np.round(iaq.score, 1)}")
print(f"  Compliant:   {iaq.compliant}  (CO₂ <= threshold)")

Acoustic (NC-35, threshold = 41.0 dBA):
  L_Aeq:      [48.7 47.7 41.7 46.2 45.6 46.1 49.4 46.1] dB
  Score:       [11.4 16.4 46.6 24.2 27.1 24.5  8.2 24.4]
  Compliant:   [False False False False False False False False]  (L_Aeq <= threshold)

IAQ (threshold 'good' = 1000.0 ppm):
  CO₂:        [768. 707. 729. 763. 554. 668. 653. 636.] ppm
  Score:       [70.  75.3 73.4 70.4 88.4 78.6 79.9 81.4]
  Compliant:   [ True  True  True  True  True  True  True  True]  (CO₂ <= threshold)


## 3. Global IEQ Index

`calculate_global_ieq()` combines all domain results into a single 0–100 index using configurable weights. Missing domains are handled by renormalizing weights.

In [6]:
from comfortpy import calculate_global_ieq

# Default weights: thermal 40%, IAQ 25%, visual 20%, acoustic 15%
# (Pierson et al. 2019)
global_result = calculate_global_ieq(
    thermal=thermal,
    visual=visual,
    acoustic=acoustic,
    iaq=iaq,
)

print(f"Global IEQ Index: {np.round(global_result.index, 1)}")
print(f"\nPer-domain scores:")
for domain, score in global_result.domain_scores.items():
    print(f"  {domain:12s}: mean={np.mean(score):.1f}, range=[{np.min(score):.1f}, {np.max(score):.1f}]")

print(f"\nWeights used: {global_result.weights_used}")
print(f"Domains:      {global_result.domains}")
print(f"Timestamps:   {global_result.n_timestamps}")

# Interpretation
mean_index = np.mean(global_result.index)
print(f"\nMean IEQ Index: {mean_index:.1f}")
if mean_index >= 80:
    print("  -> Excellent comfort conditions")
elif mean_index >= 60:
    print("  -> Good comfort conditions")
elif mean_index >= 40:
    print("  -> Moderate comfort — some domains need attention")
else:
    print("  -> Poor comfort — multiple domains below standard")

Global IEQ Index: [67.5 63.4 78.  74.6 69.3 68.6 67.5 71.3]

Per-domain scores:
  thermal     : mean=73.8, range=[63.4, 83.6]
  visual      : mean=88.9, range=[81.6, 99.7]
  acoustic    : mean=22.9, range=[8.2, 46.6]
  iaq         : mean=77.2, range=[70.0, 88.4]

Weights used: {'thermal': 0.4, 'visual': 0.2, 'acoustic': 0.15, 'iaq': 0.25}
Domains:      ['thermal', 'visual', 'acoustic', 'iaq']
Timestamps:   8

Mean IEQ Index: 70.0
  -> Good comfort conditions


## 4. Weighting Presets

ComfortPy ships with presets for common building types. The same sensor data can yield different IEQ indices depending on which domains matter most for the use case.

In [7]:
from comfortpy.integration.weights import preset_weights, WEIGHT_PRESETS

print("Available presets:")
for name, weights in WEIGHT_PRESETS.items():
    print(f"  {name:12s}: T={weights['thermal']:.0%}  V={weights['visual']:.0%}"
          f"  A={weights['acoustic']:.0%}  IAQ={weights['iaq']:.0%}")

print("\nComparing presets on the same data:")
print(f"  {'Preset':<12s} {'Mean IEQ':>10s}")
print(f"  {'-' * 24}")

results = {}
for preset_name in ["default", "equal", "office", "school", "healthcare"]:
    weights = preset_weights(preset_name)
    result = calculate_global_ieq(
        thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq,
        weights=weights,
    )
    results[preset_name] = result
    print(f"  {preset_name:<12s} {np.mean(result.index):>10.1f}")

print("\n'healthcare' weights IAQ highest (40%), 'office' weights thermal highest (45%).")

Available presets:
  default     : T=40%  V=20%  A=15%  IAQ=25%
  equal       : T=25%  V=25%  A=25%  IAQ=25%
  school      : T=27%  V=24%  A=23%  IAQ=26%
  office      : T=45%  V=15%  A=10%  IAQ=30%
  healthcare  : T=25%  V=15%  A=20%  IAQ=40%

Comparing presets on the same data:
  Preset         Mean IEQ
  ------------------------
  default            70.0
  equal              65.7
  office             72.0
  school             66.6
  healthcare         67.2

'healthcare' weights IAQ highest (40%), 'office' weights thermal highest (45%).


## 5. Compliance & Performance Contracts

`calculate_compliance()` computes what percentage of timestamps met a threshold IEQ Index. The resulting `ComplianceReport` can be exported as JSON or as a Solidity-ready contract payload.

In [8]:
from comfortpy import calculate_compliance

report = calculate_compliance(global_result, threshold=60.0)

print(f"Compliance Report:")
print(f"  Threshold:         IEQ Index >= {report.threshold}")
print(f"  Compliance rate:   {report.compliance_rate_pct:.1f}%")
print(f"  IEQ avg:           {report.ieq_index_avg:.1f}")
print(f"  IEQ std:           {report.ieq_index_std:.1f}")
print(f"  IEQ min/max:       {report.ieq_index_min:.1f} / {report.ieq_index_max:.1f}")
print(f"  Compliant hours:   {report.compliant_hours:.1f} / {report.total_hours:.1f}")

print(f"\nPer-domain compliance:")
for domain, rate in report.domain_compliance.items():
    print(f"  {domain:12s}: {rate:.1f}%")

Compliance Report:
  Threshold:         IEQ Index >= 60.0
  Compliance rate:   100.0%
  IEQ avg:           70.0
  IEQ std:           4.3
  IEQ min/max:       63.4 / 78.0
  Compliant hours:   8.0 / 8.0

Per-domain compliance:
  thermal     : 25.0%
  visual      : 100.0%
  acoustic    : 0.0%
  iaq         : 25.0%


In [9]:
# Export as JSON (ready for storage or smart contract oracle)
import json

json_str = report.to_json()
print(f"JSON report ({len(json_str)} chars):\n")
print(json.dumps(json.loads(json_str), indent=2)[:500], "...")

JSON report (762 chars):

{
  "period_start": 1783077261.317834,
  "period_end": 1783106061.317834,
  "ieq_index_avg": 70.03563237623746,
  "ieq_index_min": 63.383498732426816,
  "ieq_index_max": 77.9940353365138,
  "ieq_index_std": 4.264111229089215,
  "compliance_rate_pct": 100.0,
  "threshold": 60.0,
  "total_hours": 8.0,
  "compliant_hours": 8.0,
  "domain_compliance": {
    "thermal": 25.0,
    "visual": 100.0,
    "acoustic": 0.0,
    "iaq": 25.0
  },
  "domain_scores_avg": {
    "thermal": 73.82,
    "visual": 88. ...


In [10]:
# Export as contract payload (maps to Solidity function parameters)
payload = report.to_contract_payload()
print("Contract payload fields:")
for key, value in payload.items():
    print(f"  {key}: {value}")

Contract payload fields:
  periodStart: 1783077261
  periodEnd: 1783106061
  ieqIndexAvg: 70
  complianceRatePct: 100
  thermalCompliant: False
  visualCompliant: True
  acousticCompliant: False
  iaqCompliant: False
  totalOccupiedHours: 8
  compliantHours: 8


## 6. Missing Domain Handling

If you don't have sensors for every domain, ComfortPy renormalizes the weights so the index is still meaningful.

In [11]:
# Only thermal + IAQ available (no visual or acoustic sensors)
partial = calculate_global_ieq(thermal=thermal, iaq=iaq)

print("Only thermal + IAQ provided:")
print(f"  IEQ Index:  {np.round(partial.index, 1)}")
print(f"  Weights:    {partial.weights_used}")
print(f"  Domains:    {partial.domains}")
print("  -> Weights renormalized so thermal + IAQ sum to 1.0")

# Single domain only
thermal_only = calculate_global_ieq(thermal=thermal)
print(f"\nThermal only:")
print(f"  IEQ Index:  {np.round(thermal_only.index, 1)}")
print(f"  Weights:    {thermal_only.weights_used}")

Only thermal + IAQ provided:
  IEQ Index:  [75.1 68.5 79.6 78.5 73.  74.8 76.8 74.4]
  Weights:    {'thermal': 0.6153846153846154, 'iaq': 0.3846153846153846}
  Domains:    ['thermal', 'iaq']
  -> Weights renormalized so thermal + IAQ sum to 1.0

Thermal only:
  IEQ Index:  [78.3 64.3 83.6 83.6 63.4 72.5 74.9 70. ]
  Weights:    {'thermal': 1.0}


## Next Steps

- Try the **advanced tutorial**: `examples/advanced_domains_tutorial.ipynb`
- Read the full **User Guide**: `GUIDE.md`
- Install advanced extras: `pip install comfortpy[full]`

# This cell intentionally left blank — delete it.